# Задание 6. Извлечение признаков из текста

Перед запуском выполните `python -m ml.split`. Векторизаторы обучаются **только на train**, а val и test проходят через `transform`.

Выборка test здесь нужна только для проверки размерности матрицы: предсказания на ней не считаются.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
ROOT = ROOT if (ROOT / "manage.py").exists() else ROOT.parent
sys.path.insert(0, str(ROOT))

from ml.features import MANUAL_FEATURES, build_feature_pipeline
from ml.split import load_parts

parts = load_parts()
COLUMNS = ["text_raw", "text_clean"]

summary = pd.DataFrame({
    name: {"записей": len(part), "spam": int(part["label"].sum()), "доля spam, %": round(part["label"].mean() * 100, 2)}
    for name, part in parts.items()
}).T
summary

,записей,spam,"доля spam, %"
train,3595.0,435.0,12.10
val,771.0,93.0,12.06
test,771.0,93.0,12.06


## Обучение на train, преобразование val и test

In [2]:
features = build_feature_pipeline()
X_train = features.fit_transform(parts["train"][COLUMNS])  # единственный fit
X_val = features.transform(parts["val"][COLUMNS])
X_test = features.transform(parts["test"][COLUMNS])

union = features.named_steps["features"]
n_word = len(union.named_transformers_["word"].vocabulary_)
n_char = len(union.named_transformers_["char"].vocabulary_)

print("Тип матрицы:", type(X_train).__name__)
print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Признаки: слова {n_word}, символы {n_char}, ручные {len(MANUAL_FEATURES)}, всего {n_word + n_char + len(MANUAL_FEATURES)}")
print(f"Заполненность X_train: {X_train.nnz / (X_train.shape[0] * X_train.shape[1]):.3%}")

assert X_train.shape[0] == len(parts["train"])
assert X_val.shape[0] == len(parts["val"])
assert X_test.shape[0] == len(parts["test"])
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]

Тип матрицы: csr_matrix
X_train: (3595, 34107)
X_val:   (771, 34107)
X_test:  (771, 34107)
Признаки: слова 9028, символы 25072, ручные 7, всего 34107
Заполненность X_train: 0.433%


## Топ-15 признаков со средним TF-IDF по классам

Расчёт выполнен только на train. Слова и символьные N-граммы показаны отдельно, потому что их веса имеют разный масштаб.

In [3]:
names = np.array(union.get_feature_names_out())
labels = parts["train"]["label"].to_numpy()


def top_features(prefix, label, n=15):
    mask = np.char.startswith(names.astype(str), prefix)
    columns = np.where(mask)[0]
    means = np.asarray(X_train[labels == label][:, columns].mean(axis=0)).ravel()
    order = means.argsort()[::-1][:n]
    return [f"{names[columns[i]].removeprefix(prefix)!r} ({means[i]:.3f})" for i in order]


pd.DataFrame({
    "слова, ham": top_features("word__", 0),
    "слова, spam": top_features("word__", 1),
    "символы, ham": top_features("char__", 0),
    "символы, spam": top_features("char__", 1),
}, index=range(1, 16))

,"слова, ham","слова, spam","символы, ham","символы, spam"
1,'you' (0.031),'<num>' (0.077),' th' (0.019),'one' (0.037)
2,'?' (0.029),'<phone>' (0.053),' yo' (0.017),'num' (0.035)
3,'to' (0.024),'to' (0.048),' i ' (0.017),'num>' (0.035)
4,'the' (0.020),'<money>' (0.046),'you' (0.016),'um>' (0.035)
5,'<num>' (0.020),'!' (0.043),' you' (0.016),'<num>' (0.035)
6,'in' (0.019),'call' (0.040),'ing' (0.016),'<nu' (0.035)
7,'me' (0.018),'your' (0.033),'ng ' (0.015),'<num' (0.035)
8,'and' (0.017),'to <num>' (0.033),'ing ' (0.015),' <nu' (0.033)
9,'it' (0.016),'call <phone>' (0.031),' to' (0.015),' <n' (0.033)
10,'my' (0.016),'free' (0.031),'ou ' (0.014),' <num' (0.033)


## Ручные признаки на train

In [4]:
extract = union.named_transformers_["manual"].named_steps["extract"]
manual = pd.DataFrame(extract.transform(parts["train"][COLUMNS]), columns=MANUAL_FEATURES)
manual["класс"] = np.where(labels == 1, "spam", "ham")
manual.groupby("класс").mean().round(3).T

класс,ham,spam
length_chars,70.419,138.694
upper_ratio,0.076,0.161
exclamation_count,0.172,0.683
has_url,0.003,0.209
has_money,0.003,0.370
has_phone,0.000,0.499
rule_group_count,0.022,0.393


Объяснение назначения признаков находится в файле `docs/06_features.md`.